# Activation Functions — the nonlinearity that makes depth matter

> Tutorial pair for [`activations.py`](activations.py).

## 1. Intuition
Stack only linear layers and you get... another linear layer: $W_2(W_1x)=(W_2W_1)x$.
The **activation** is the pointwise nonlinearity squeezed between affine maps; it is
the entire reason a deep network can represent something a single matrix cannot.
Its *derivative* is equally important: backprop multiplies by $f'$ at **every** layer,
so the shape of $f'$ decides whether gradients survive depth or **vanish**.

## 2. Concept (the slide)
- A layer computes $a = f(z),\ z = aW + b$. The choice of $f$ trades off:
  - **Range / centering** (sigmoid is in $(0,1)$ and not zero-centred; tanh is in $(-1,1)$).
  - **Saturation** — where $f'\!\to 0$. A saturated unit passes *no* gradient.
  - **Smoothness** — ReLU has a kink; GELU/Swish are smooth, which helps optimization.
- **Saturating** (sigmoid, tanh): $f'$ bounded well below 1 over most of the input range.
- **Non-saturating** (ReLU family, GELU, Swish): $f'\approx 1$ on the active region.
- **Softmax** is the odd one out: vector-valued, used at the output to make a distribution;
  its derivative is a full **Jacobian**, not a scalar.

## 3. Math derivation — each function and its derivative

**Sigmoid.** $\sigma(x)=\dfrac{1}{1+e^{-x}}$. Differentiate:
$$\sigma'(x)=\frac{e^{-x}}{(1+e^{-x})^2}=\sigma(x)\bigl(1-\sigma(x)\bigr)\le\tfrac14.$$
The bound $\tfrac14$ is the crux: each layer scales the gradient by at most $0.25$.

**Tanh.** $\tanh(x)=\dfrac{e^x-e^{-x}}{e^x+e^{-x}}$, and
$$\tanh'(x)=1-\tanh^2(x)\in[0,1],$$
with max $1$ at $x=0$ but $\to 0$ for $|x|$ large (still saturates, but zero-centred).

**ReLU.** $\mathrm{ReLU}(x)=\max(0,x)$, so
$$\mathrm{ReLU}'(x)=\begin{cases}1 & x>0\\ 0 & x<0\end{cases}$$
(undefined at $0$; we take $0$). No shrinkage on the active side — but a unit stuck at
$x<0$ is **dead** (zero gradient forever).

**LeakyReLU.** $f(x)=x$ for $x>0$, else $\alpha x$; derivative $1$ or $\alpha$. The small
$\alpha$ keeps negatives alive, fixing dead units.

**ELU.** $f(x)=x$ for $x>0$, else $\alpha(e^x-1)$. Then
$$f'(x)=\begin{cases}1 & x>0\\ \alpha e^x = f(x)+\alpha & x\le 0\end{cases}$$
— smooth, negative-saturating to $-\alpha$ (pushes mean activations toward 0).

**GELU** (tanh approximation, used in BERT/GPT). With
$u(x)=\sqrt{2/\pi}\,(x+0.044715x^3)$,
$$\mathrm{GELU}(x)=\tfrac12 x\,(1+\tanh u).$$
Product + chain rule, using $u'(x)=\sqrt{2/\pi}\,(1+3\cdot0.044715\,x^2)$ and
$\frac{d}{du}\tanh u = 1-\tanh^2u$:
$$\mathrm{GELU}'(x)=\tfrac12(1+\tanh u)+\tfrac12 x\,(1-\tanh^2u)\,u'(x).$$

**Swish / SiLU.** $f(x)=x\,\sigma(\beta x)$ ($\beta=1$ is SiLU). Product rule:
$$f'(x)=\sigma(\beta x)+\beta x\,\sigma(\beta x)\bigl(1-\sigma(\beta x)\bigr).$$
Note $f'$ can exceed $1$ — Swish/GELU are **non-monotone** and slightly self-gating.

**Softmax** (vector-valued). $s_i=\dfrac{e^{z_i}}{\sum_j e^{z_j}}$. The Jacobian:
$$\frac{\partial s_i}{\partial z_j}=s_i(\delta_{ij}-s_j)\;\Longrightarrow\;
  J=\operatorname{diag}(s)-ss^\top.$$
For backprop you never form $J$; the **vector-Jacobian product** is
$$(g^\top J)_j = s_j\Bigl(g_j-\textstyle\sum_k g_k s_k\Bigr).$$

### The vanishing-gradient link
Backprop through $L$ layers multiplies $L$ copies of $f'$ (times weight factors). With
sigmoid, $\prod f' \le 0.25^L \to 0$ geometrically — early layers stop learning. ReLU/GELU/Swish
keep $f'\approx 1$, so the signal survives. We **measure** exactly this below.

## 4. NumPy implementation — forward + analytic derivative for each

In [ ]:
# ===== actual implementation from activations.py =====
from __future__ import annotations

import numpy as np

SEED = 0

class Activation:
    """Base class: subclasses define forward `f(x)` and elementwise `df(x)`.

    Convention: `df` returns df/dx evaluated at the *input* x (not at f(x)),
    which is what the backward pass multiplies: delta_in = delta_out * df(x).
    """

    name = "activation"

    def forward(self, x: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    def df(self, x: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    # convenience aliases
    def __call__(self, x): return self.forward(x)

class Sigmoid(Activation):
    r"""sigma(x) = 1/(1+e^{-x});  sigma'(x) = sigma(x)(1-sigma(x)) in [0, 1/4]."""
    name = "sigmoid"

    def forward(self, x):
        # clip to avoid overflow in exp; output in (0,1)
        return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))

    def df(self, x):
        s = self.forward(x)
        return s * (1.0 - s)

class Tanh(Activation):
    r"""tanh(x);  tanh'(x) = 1 - tanh(x)^2 in [0, 1]."""
    name = "tanh"

    def forward(self, x):
        return np.tanh(x)

    def df(self, x):
        return 1.0 - np.tanh(x) ** 2

class ReLU(Activation):
    r"""ReLU(x) = max(0,x);  ReLU'(x) = 1 if x>0 else 0 (undefined at 0 -> 0)."""
    name = "relu"

    def forward(self, x):
        return np.maximum(0.0, x)

    def df(self, x):
        return (x > 0).astype(x.dtype)

class LeakyReLU(Activation):
    r"""LeakyReLU(x) = x if x>0 else alpha*x;  derivative 1 or alpha."""
    name = "leaky_relu"

    def __init__(self, alpha: float = 0.01):
        self.alpha = alpha

    def forward(self, x):
        return np.where(x > 0, x, self.alpha * x)

    def df(self, x):
        return np.where(x > 0, 1.0, self.alpha)

class ELU(Activation):
    r"""ELU(x) = x if x>0 else alpha*(e^x - 1).
    Derivative: 1 if x>0 else alpha*e^x = ELU(x)+alpha on the negative side."""
    name = "elu"

    def __init__(self, alpha: float = 1.0):
        self.alpha = alpha

    def forward(self, x):
        return np.where(x > 0, x, self.alpha * (np.exp(np.clip(x, -50, 50)) - 1.0))

    def df(self, x):
        return np.where(x > 0, 1.0, self.alpha * np.exp(np.clip(x, -50, 50)))

class GELU(Activation):
    r"""GELU(x) = x * Phi(x), Phi = standard normal CDF.
    We use the tanh approximation (as in the original paper / BERT/GPT):
        GELU(x) ~= 0.5 x (1 + tanh[ sqrt(2/pi) (x + 0.044715 x^3) ]).
    Derivative obtained by the product + chain rule on that closed form."""
    name = "gelu"
    _c = np.sqrt(2.0 / np.pi)              # sqrt(2/pi)
    _a = 0.044715

    def _inner(self, x):
        return self._c * (x + self._a * x ** 3)

    def forward(self, x):
        return 0.5 * x * (1.0 + np.tanh(self._inner(x)))

    def df(self, x):
        u = self._inner(x)
        t = np.tanh(u)
        # du/dx = c (1 + 3 a x^2)
        du = self._c * (1.0 + 3.0 * self._a * x ** 2)
        sech2 = 1.0 - t ** 2               # d/du tanh(u)
        # product rule on 0.5 x (1 + tanh u)
        return 0.5 * (1.0 + t) + 0.5 * x * sech2 * du

class Swish(Activation):
    r"""Swish / SiLU(x) = x * sigma(beta*x).  With beta=1 this is SiLU.
    Derivative: sigma(bx) + x * beta * sigma(bx)(1 - sigma(bx))
              = sigma(bx) + b*x*sigma(bx) - b*x*sigma(bx)^2
              = beta*Swish(x) + sigma(bx)(1 - beta*Swish(x))."""
    name = "swish"

    def __init__(self, beta: float = 1.0):
        self.beta = beta

    def _sig(self, x):
        return 1.0 / (1.0 + np.exp(-np.clip(self.beta * x, -50, 50)))

    def forward(self, x):
        return x * self._sig(x)

    def df(self, x):
        s = self._sig(x)
        return s + self.beta * x * s * (1.0 - s)

class Softmax:
    r"""Vector-valued activation over the last axis.
        softmax(z)_i = e^{z_i} / sum_j e^{z_j}  (shift by max for stability).
    Its Jacobian for a single sample is
        J_ij = s_i (delta_ij - s_j),
    i.e. diag(s) - s s^T. We expose both the forward and the per-sample Jacobian.
    """
    name = "softmax"

    def forward(self, z: np.ndarray) -> np.ndarray:
        z = z - z.max(axis=-1, keepdims=True)
        e = np.exp(z)
        return e / e.sum(axis=-1, keepdims=True)

    def __call__(self, z): return self.forward(z)

    def jacobian(self, z: np.ndarray) -> np.ndarray:
        """Batched Jacobian, shape (N, K, K) for input (N, K)."""
        s = self.forward(z)                          # (N, K)
        N, K = s.shape
        diag = np.einsum("nk,kj->nkj", s, np.eye(K))  # diag(s)
        outer = np.einsum("ni,nj->nij", s, s)         # s s^T
        return diag - outer

    def vjp(self, z: np.ndarray, g: np.ndarray) -> np.ndarray:
        """Vector-Jacobian product g @ J without forming J: efficient backward.
        dL/dz = s * (g - sum_j g_j s_j)."""
        s = self.forward(z)
        return s * (g - (g * s).sum(axis=-1, keepdims=True))

## 5. PyTorch implementation — same functions; derivatives via autograd

In [ ]:
# ===== actual implementation from activations.py =====
ELEMENTWISE = {
    "sigmoid": Sigmoid(),
    "tanh": Tanh(),
    "relu": ReLU(),
    "leaky_relu": LeakyReLU(0.1),
    "elu": ELU(1.0),
    "gelu": GELU(),
    "swish": Swish(1.0),
}

def check_gradient(act: Activation, x: np.ndarray, eps: float = 1e-5) -> float:
    """Max abs error between analytic df and a central finite difference.
    Numerically verifies every hand-derived derivative above."""
    num = (act.forward(x + eps) - act.forward(x - eps)) / (2 * eps)
    return float(np.max(np.abs(num - act.df(x))))

import torch

import torch.nn as nn

import torch.nn.functional as F

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

TORCH_ACT = {
    "sigmoid": torch.sigmoid,
    "tanh": torch.tanh,
    "relu": F.relu,
    "leaky_relu": lambda x: F.leaky_relu(x, negative_slope=0.1),
    "elu": lambda x: F.elu(x, alpha=1.0),
    "gelu": lambda x: F.gelu(x, approximate="tanh"),
    "swish": F.silu,                       # SiLU == Swish(beta=1)
    "softmax": lambda x: F.softmax(x, dim=-1),
}

def torch_forward_and_grad(name: str, x_np: np.ndarray):
    """Run the torch activation and obtain its derivative via autograd.
    Returns (forward, derivative) as numpy arrays, on the chosen device."""
    dev = get_device()
    x = torch.tensor(x_np, dtype=torch.float64, device=dev, requires_grad=True)
    y = TORCH_ACT[name](x)
    # sum() so we get dy_i/dx_i on the diagonal for elementwise functions
    y.sum().backward()
    return y.detach().cpu().numpy(), x.grad.detach().cpu().numpy()

def demo():
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    x = np.linspace(-6, 6, 2001)
    # grid that avoids the kink at x=0 (where ReLU/LeakyReLU are non-differentiable
    # and a central finite difference straddles the corner)
    x_smooth = np.linspace(-6, 6, 2000) + 1e-3

    # (a) verify every analytic derivative against finite differences
    print("Analytic derivative vs finite-difference (max abs error):")
    for nm, act in ELEMENTWISE.items():
        err = check_gradient(act, x_smooth)
        print(f"  {nm:11s}: {err:.2e}")

    # (b) cross-check NumPy forward & derivative against PyTorch autograd
    print("\nNumPy vs PyTorch (max abs error on f and f'):")
    for nm, act in ELEMENTWISE.items():
        tf, tg = torch_forward_and_grad(nm, x)
        ef = float(np.max(np.abs(act.forward(x) - tf)))
        eg = float(np.max(np.abs(act.df(x) - tg)))
        print(f"  {nm:11s}: |df_f|={ef:.2e}  |df_grad|={eg:.2e}")

    # softmax forward agreement
    z = np.random.randn(5, 4)
    sm = Softmax()
    tsm, _ = torch_forward_and_grad("softmax", z)
    print(f"  softmax    : |df_f|={np.max(np.abs(sm.forward(z) - tsm)):.2e}")
    # verify the efficient VJP against the explicit Jacobian
    g = np.random.randn(*z.shape)
    vjp_fast = sm.vjp(z, g)
    vjp_full = np.einsum("ni,nij->nj", g, sm.jacobian(z))
    print(f"  softmax vjp vs Jacobian: {np.max(np.abs(vjp_fast - vjp_full)):.2e}")

    # (c) MEASURE saturation — the vanishing-gradient link
    print("\nSaturation report (root cause of vanishing gradients):")
    print("  activation   max f'    mean f' over [-6,6]   %|f'|<0.01 (saturated/dead)")
    for nm, act in ELEMENTWISE.items():
        d = act.df(x)
        dead = 100.0 * np.mean(np.abs(d) < 0.01)
        print(f"  {nm:11s} {d.max():7.3f}   {d.mean():10.3f}            {dead:6.1f}%")
    print("  -> sigmoid/tanh: max f' <= 1 and most inputs saturate, so a product")
    print("     of L such factors shrinks geometrically => VANISHING gradients.")
    print("     ReLU family/GELU/Swish keep f' ~ 1 on the active region => deep")
    print("     nets stay trainable (see training-techniques/README.md).")

## 6. Run — verify every derivative & **measure saturation**

In [ ]:
demo()

## 7. Visualization — each activation and its derivative

Top row: the activations. Bottom row: their derivatives. Watch how sigmoid/tanh
derivatives collapse toward 0 away from the origin (saturation), while the ReLU
family / GELU / Swish keep their derivative near 1 on the active side.

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import activations as M

x = np.linspace(-6, 6, 600)
acts = M.ELEMENTWISE
fig, axes = plt.subplots(2, len(acts), figsize=(2.1*len(acts), 4.6), sharex=True)
for j, (nm, act) in enumerate(acts.items()):
    axes[0, j].plot(x, act.forward(x), color="C0")
    axes[0, j].set_title(nm); axes[0, j].grid(True, alpha=.3); axes[0, j].axhline(0, color="k", lw=.5)
    axes[1, j].plot(x, act.df(x), color="C3")
    axes[1, j].grid(True, alpha=.3); axes[1, j].axhline(0, color="k", lw=.5)
axes[0, 0].set_ylabel("f(x)"); axes[1, 0].set_ylabel("f'(x)")
fig.suptitle("Activations (top) and their derivatives (bottom)")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- The derivative $f'$ is what backprop propagates — its shape *is* the trainability story.
- **Sigmoid** saturates hard ($f'\le0.25$) and is not zero-centred → avoid in hidden layers.
- **ReLU** is cheap and non-saturating but can produce **dead units**; LeakyReLU/ELU fix that.
- **GELU/Swish** are smooth, non-monotone, and dominate modern Transformers/CNNs.
- **Softmax** belongs at the *output*; pair it with cross-entropy so the gradient
  simplifies to $a-y$ (see [`mlp.ipynb`](mlp.ipynb)).
- Vanishing gradients motivate ReLU/He init, normalization, and residuals — see
  `06.training-techniques/README.md`.